In [1]:
%env HSA_OVERRIDE_GFX_VERSION=10.3.0
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from copy import deepcopy
import re

env: HSA_OVERRIDE_GFX_VERSION=10.3.0


In [2]:
def create_atomic_electron_config_database():
    df_pde = pd.read_csv('../datasets/PubChemElements_all.csv', usecols=['Symbol', 'ElectronConfiguration'])
    num_elements = len(df_pde['Symbol'])
    max_shell_config = {'1s': 0, 
                        '2s': 0, 
                        '2p': 0, 
                        '3s': 0, 
                        '3p': 0, 
                        '4s': 0, 
                        '3d': 0, 
                        '4p': 0, 
                        '5s': 0, 
                        '4d': 0, 
                        '5p': 0, 
                        '6s': 0, 
                        '4f': 0, 
                        '5d': 0, 
                        '6p': 0, 
                        '7s': 0, 
                        '5f': 0, 
                        '6d': 0, 
                        '7p': 0}
    outer_shell_config = {'He':0, 'Ne':0, 'Ar':0, 'Kr':0, 'Xe':0, 'Rn':0, 's':0, 'p':0, 'd':0, 'f':0}
    num_max_shells = len(max_shell_config)
    electronic_database = np.zeros((num_elements,num_max_shells))
    compact_electronic_config_db = np.zeros((num_elements, len(outer_shell_config), len(outer_shell_config)))
    database_dict = {}
    for i in range(num_elements):
        atomic_electron_config = {}
        outer_shell = deepcopy(outer_shell_config)
        core_match = re.search(r'\[([A-Za-z]+)\]', df_pde['ElectronConfiguration'].iloc[i])
        if core_match:
            core_element = core_match.group(1)
            atomic_electron_config |= database_dict[core_element]
            outer_shell[core_element] = 1
        matches = re.findall(r'(\d[spdfg])(\d+)', df_pde['ElectronConfiguration'].iloc[i])
        for shell, electrons in matches:
            atomic_electron_config[shell] = int(electrons)
            outer_shell[shell[1]] = int(electrons)
        database_dict[df_pde['Symbol'].iloc[i]] = atomic_electron_config
        template = deepcopy(max_shell_config)
        for shell in atomic_electron_config:
            template[shell] = atomic_electron_config[shell]
        for shellnum in range(num_max_shells):
            electronic_database[i, shellnum] = list(template.values())[shellnum]
        compact_config = np.array(list(outer_shell.values()))
        for compact_db_index in range(len(outer_shell_config)):
            compact_electronic_config_db[i, 0, compact_db_index] = compact_config[compact_db_index]
        for k in range(1, len(outer_shell_config)):
            compact_electronic_config_db[i, k] = np.roll(compact_electronic_config_db[i, k-1], 1)
    return compact_electronic_config_db, electronic_database, database_dict


def formula_to_molecular_img(formula, atomic_img_db):
    df_pde = pd.read_csv('../datasets/PubChemElements_all.csv', usecols=['AtomicNumber', 'Symbol'])
    matches = re.findall(r'([A-Z][a-z]*)(\d*)', formula)
    molecular_img = []
    for element, count in matches:
        # If a number exists, use it; otherwise default to 1
        # 'V2' -> count is '2', 'Sc' -> count is '' (which becomes 1)
        n = int(count) if count else 1
        molecular_img.extend([atomic_img_db[df_pde.loc[df_pde['Symbol'] == element]['AtomicNumber'].item()]] * n)
        # formula_extended.extend([element] * n)
        # print(formula_extended)
    if len(molecular_img) < 4:
        molecular_img.append(np.zeros((10,10)))
    # for element in formula_extended:
    #     atomic_num_list.append(element_dict[element])
    return np.array(molecular_img)

In [3]:
df = pd.read_csv('../datasets/heusler_magnetic.csv', usecols=['formula', 'heusler type', 'num_electron', 'struct type', 'latt const', 'tetragonality', 'e_form', 'pol fermi', 'gap width', 'stability', 'mu_b', 'mu_b saturation'])

heusler_map = ['Full Heusler', 'Half Heusler', 'Inverse Heusler']
for typ in heusler_map:
    df[typ] = (df['heusler type'] == typ)

df['struct type'].replace('Tetragonal', 'tetragonal', inplace=True)
struct_map = {'D022':[1,0,0,0,0,0], 'L21':[0,1,0,0,0,0], 'C1b':[0,0,1,0,0,0], 'tetragonal':[0,0,0,1,0,0], 'triclinic':[0,0,0,0,1,0], 'Xa':[0,0,0,0,0,1]}
df['struct type'] = df['struct type'].map(struct_map)

gap_width_arr = pd.to_numeric(df['gap width'], errors='coerce')
df['gap width'] = gap_width_arr.fillna(0)

pol_fermi = pd.to_numeric(df['pol fermi'], errors='coerce')
df['pol fermi'] = pol_fermi.fillna(0)

e_form = pd.to_numeric(df['e_form'], errors='coerce')
# df['e_form'] = e_form.fillna(100)

mbs = pd.to_numeric(df['mu_b saturation'], errors='coerce')
mbs[np.isnan(mbs)] = 0
df['mu_b saturation'] = mbs

# with pd.option_context("future.no_silent_downcasting", True):
#     df['stability'] = df['stability'].fillna(False).astype(bool)

# instead of replacing values and poisoning the data, we simply clean it all up
df = df.dropna()
df['stability'] = df['stability'].astype(bool)

atomic_img_db, _, _ = create_atomic_electron_config_database()
molecular_img_lst = [formula_to_molecular_img(x, atomic_img_db) for x in df['formula']]
e_form_max = df['e_form'].max()
latt_sz_max = df['latt const'].max()
# print(x_train.shape)

x_train_up = np.array(molecular_img_lst, dtype=np.float32)
y_train_eform_up = df['e_form'].to_numpy(dtype=np.float32).reshape((1081,1))/e_form_max
y_train_latsz_up = df['latt const'].to_numpy(dtype=np.float32).reshape((1081,1))/latt_sz_max
y_train_stabi_up = df['stability'].to_numpy(dtype=np.float32).reshape((1081,1))
# y_train_heuty_up = df['heusler_type'].to_numpy(dtype=np.float32).reshape((1081,3))

x_train, x_test, y_train_eform, y_test_eform, y_train_latsz, y_test_latsz, y_train_stabi, y_test_stabi = train_test_split(x_train_up, 
                            y_train_eform_up, 
                            y_train_latsz_up, 
                            y_train_stabi_up, 
                            test_size=0.2, random_state=3, shuffle=True)
# df['formula'] = molecular_img_lst
x_train, x_test, y_train_eform, y_test_eform, y_train_latsz, y_test_latsz, y_train_stabi, y_test_stabi = torch.tensor(x_train, device="cuda"), torch.tensor(x_test, device="cuda"), torch.tensor(y_train_eform, device="cuda"), torch.tensor(y_test_eform, device="cuda"), torch.tensor(y_train_latsz, device="cuda"), torch.tensor(y_test_latsz, device="cuda"), torch.tensor(y_train_stabi, device="cuda"), torch.tensor(y_test_stabi, device="cuda")
# x_train, x_test, y_train_eform, y_test_eform, y_train_latsz, y_test_latsz, y_train_stabi, y_test_stabi = torch.Tensor(x_train), torch.Tensor(x_test), torch.Tensor(y_train_eform), torch.Tensor(y_test_eform), torch.Tensor(y_train_latsz), torch.Tensor(y_test_latsz), torch.Tensor(y_train_stabi), torch.Tensor(y_test_stabi)

print(x_train.shape, y_train_eform.shape, y_train_stabi.shape, y_train_latsz.shape)
print(x_test.shape, y_test_eform.shape, y_test_stabi.shape, y_test_latsz.shape)
# df.to_csv('processed_heusler_data.csv')
# training_df = df.drop(columns=['stability'], inplace=False)
# predict_arr = df['stability'].to_numpy()
# print(training_df['gap width'].unique())
# print(formula_to_atomic_num_lst)

torch.Size([864, 4, 10, 10]) torch.Size([864, 1]) torch.Size([864, 1]) torch.Size([864, 1])
torch.Size([217, 4, 10, 10]) torch.Size([217, 1]) torch.Size([217, 1]) torch.Size([217, 1])


In [11]:
class MyCNNModel(nn.Module):
    def __init__(self):
        super(MyCNNModel, self).__init__()
        self.convlayer_1 = nn.Conv2d(4, 10, 2)
        # self.convlayer_2 = nn.Conv2d(8, 16, 4)
        # self.deeplayer_1 = nn.Linear(16*4*4, 256)
        self.deeplayer_1 = nn.Linear(10*9*9, 256)
        self.deeplayer_2 = nn.Linear(256, 32)
        
        self.deeplayer_3_stability = nn.Linear(32, 1) # stability
        self.deeplayer_3_lattconst = nn.Linear(32, 1) # lattice_const
        self.deeplayer_3_formation = nn.Linear(32, 1) # formation energy

    def forward(self, x):
        x = nn.functional.relu(self.convlayer_1(x))
        # x = nn.functional.relu(self.convlayer_2(x))
        x = torch.flatten(x, 1)
        x = nn.functional.relu(self.deeplayer_1(x))
        x = nn.functional.relu(self.deeplayer_2(x))
        
        stability = nn.functional.sigmoid(self.deeplayer_3_stability(x))
        lattconst = nn.functional.tanh(self.deeplayer_3_lattconst(x))
        formation = nn.functional.tanh(self.deeplayer_3_formation(x))
        # x = self.deeplayer_3(x)
        # return stability
        return stability, lattconst, formation

device = torch.device("cuda")
my_cnn_net = MyCNNModel()
my_cnn_net.to(device)
loss_fn_stab = nn.BCELoss()
loss_fn_latt = nn.MSELoss()
loss_fn_form = nn.MSELoss()
optimizer_fn = torch.optim.AdamW(my_cnn_net.parameters(), lr=0.001, weight_decay=1e-4)

# Training loop
train_epochs = 50000
min_validation_loss = 1
for epoch in range(train_epochs):
    my_cnn_net.train()
    optimizer_fn.zero_grad()
    # Forward pass
    # output_stability = my_cnn_net(x_train)
    output_stability, output_lattconst, output_formation = my_cnn_net(x_train)
    
    loss_stab = loss_fn_stab(output_stability, y_train_stabi)
    loss_latt = loss_fn_latt(output_lattconst, y_train_latsz)
    loss_form = loss_fn_form(output_formation, y_train_eform)

    # total_loss_val = (loss_stab)
    total_loss_val = (0.2*loss_stab+0.6*loss_latt+1.5*loss_form)
    
    # Backward pass and optimization
    total_loss_val.backward()
    optimizer_fn.step()

    my_cnn_net.eval()
    with torch.no_grad():
        # raw_stab = my_cnn_net(x_test)
        raw_stab, raw_latt, raw_form = my_cnn_net(x_test)
        # total_validation_loss = loss_fn_stab(raw_stab, y_test_stabi)
        total_validation_loss = 0.2*loss_fn_stab(raw_stab, y_test_stabi) + 0.6*loss_fn_latt(raw_latt, y_test_latsz) + 1.5*loss_fn_form(raw_form, y_test_eform)

    # Optional: Print loss every few epochs
    # if (total_validation_loss.item() > min_validation_loss+0.02) and (epoch > 300):
    #     break
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{train_epochs}], Loss: {total_loss_val.item():.4f}, Validation Loss: {total_validation_loss.item():.4f}')
    # min_validation_loss = min_validation_loss if (min_validation_loss < total_validation_loss.item()) else total_validation_loss.item()

Epoch [1000/50000], Loss: 0.0119, Validation Loss: 0.1048
Epoch [2000/50000], Loss: 0.0060, Validation Loss: 0.1519
Epoch [3000/50000], Loss: 0.0042, Validation Loss: 0.2543
Epoch [4000/50000], Loss: 0.0037, Validation Loss: 0.2701
Epoch [5000/50000], Loss: 0.0039, Validation Loss: 0.2768
Epoch [6000/50000], Loss: 0.0034, Validation Loss: 0.2809
Epoch [7000/50000], Loss: 0.0034, Validation Loss: 0.2818
Epoch [8000/50000], Loss: 0.0033, Validation Loss: 0.2799
Epoch [9000/50000], Loss: 0.0033, Validation Loss: 0.2801
Epoch [10000/50000], Loss: 0.0034, Validation Loss: 0.2814
Epoch [11000/50000], Loss: 0.0034, Validation Loss: 0.2818
Epoch [12000/50000], Loss: 0.0035, Validation Loss: 0.2815
Epoch [13000/50000], Loss: 0.0030, Validation Loss: 0.2788
Epoch [14000/50000], Loss: 0.0034, Validation Loss: 0.2815
Epoch [15000/50000], Loss: 0.0032, Validation Loss: 0.2745
Epoch [16000/50000], Loss: 0.0030, Validation Loss: 0.2815
Epoch [17000/50000], Loss: 0.0029, Validation Loss: 0.2826
Epoch 

In [14]:
my_cnn_net.eval()

with torch.no_grad():
    # raw_stab = my_cnn_net(x_test)
    raw_stab, raw_latt, raw_form = my_cnn_net(x_test)
    
    stability_preds = (raw_stab > 0.8).float()
    lattconst_preds = raw_latt*latt_sz_max
    eform_preds = raw_form*e_form_max
    # print(stability_preds, y_train_stabi)
    # print(lattconst_preds, y_train_latsz*latt_sz_max)
    # print(eform_preds, y_train_eform*e_form_max)
    
    error_stab = (stability_preds!=y_test_stabi)
    error_lattconst = (abs(lattconst_preds - y_test_latsz*latt_sz_max) > 0.3).float()
    error_eform = (abs(eform_preds - y_test_eform*e_form_max) > 0.1).float()
    total_data = y_test_stabi.shape[0]
    # print(f"Total Error Count For:\n  -Stability = {float(error_stab.sum())}")
    print(f"Total Error Count For:\n  -Stability = {float(error_stab.sum())}\n  -lattconst = {float(error_lattconst.sum())}\n  -e_formation = {float(error_eform.sum())}")
    # print(f"Total Accuracy For:\n  -Stability = {1-float(error_stab.sum())/total_data}")
    print(f"Total Accuracy For:\n  -Stability = {1-float(error_stab.sum())/total_data}\n  -lattconst = {1-float(error_lattconst.sum())/total_data}\n  -e_formation = {1-float(error_eform.sum())/total_data}")
    # print(f"Stability error for:\n{df['formula'][error_stab.numpy().reshape(-1)]}")
    

Total Error Count For:
  -Stability = 31.0
  -lattconst = 40.0
  -e_formation = 41.0
Total Accuracy For:
  -Stability = 0.8571428571428572
  -lattconst = 0.815668202764977
  -e_formation = 0.8110599078341014
